In [0]:
%sql
-- ============================================================================
-- 1. ROW COUNTS & DATA VOLUME DISTRIBUTION
-- ============================================================================
WITH row_counts AS (
  SELECT 'fda_master' AS table_name, 'CSV' AS format, COUNT(*) AS row_count, 12 AS expected_files 
  FROM hc_regulatory_sandbox.bronze_ingestion.fda_master
  UNION ALL
  SELECT 'providers', 'CSV', COUNT(*), 1 FROM hc_regulatory_sandbox.bronze_ingestion.providers
  UNION ALL
  SELECT 'spl_prescription', 'XML', COUNT(*), 62 FROM hc_regulatory_sandbox.bronze_ingestion.spl_labels_prescription
  UNION ALL
  SELECT 'spl_otc', 'XML', COUNT(*), 46 FROM hc_regulatory_sandbox.bronze_ingestion.spl_labels_otc
  UNION ALL
  SELECT 'spl_animal', 'XML', COUNT(*), 1 FROM hc_regulatory_sandbox.bronze_ingestion.spl_labels_animal
  UNION ALL
  SELECT 'spl_other', 'XML', COUNT(*), 1 FROM hc_regulatory_sandbox.bronze_ingestion.spl_labels_other
)
SELECT 
  rc.table_name,
  rc.format,
  rc.row_count,
  rc.expected_files,
  CASE 
    WHEN rc.format = 'XML' AND rc.row_count != rc.expected_files THEN '⚠️ MISMATCH'
    WHEN rc.format = 'CSV' THEN '✓ CSV'
    ELSE '✓ OK'
  END AS status,
  ROUND(100.0 * rc.row_count / (SELECT SUM(row_count) FROM row_counts), 2) AS pct_of_total,
  CASE 
    WHEN rc.format = 'XML' AND rc.row_count > rc.expected_files THEN CONCAT('Duplicates found: ', rc.row_count - rc.expected_files, ' extra rows')
    WHEN rc.format = 'XML' AND rc.row_count < rc.expected_files THEN CONCAT('Missing files: ', rc.expected_files - rc.row_count, ' files not ingested')
    WHEN rc.format = 'XML' THEN 'All files ingested (1 row per XML file)'
    WHEN rc.format = 'CSV' THEN CONCAT(rc.row_count, ' data rows from ', rc.expected_files, ' CSV files')
    ELSE 'OK'
  END AS data_quality_note
FROM row_counts rc
ORDER BY rc.row_count DESC;

In [0]:
%sql
-- ============================================================================
-- 2. SCHEMA VALIDATION: FDA MASTER (CSV)
-- ============================================================================
-- Verify column cleaning logic worked (special chars replaced with underscores)
DESCRIBE TABLE hc_regulatory_sandbox.bronze_ingestion.fda_master;

In [0]:
%sql
-- ============================================================================
-- 3. BINARY CONTENT VERIFICATION: XML TABLES
-- ============================================================================
-- Verify binary content is stored correctly (path, length, content)
SELECT 
  path, 
  length AS file_size_bytes,
  ROUND(length / 1024.0, 2) AS file_size_kb,
  modificationTime,
  SUBSTRING(CAST(content AS STRING), 1, 100) AS xml_preview
FROM hc_regulatory_sandbox.bronze_ingestion.spl_labels_prescription 
LIMIT 3;

In [0]:
%sql
-- ============================================================================
-- 4. DUPLICATE DETECTION: XML TABLES (CRITICAL)
-- ============================================================================
-- ⚠️ Check for duplicate file paths (indicates re-ingestion or checkpoint issues)
WITH duplicates AS (
  SELECT 'spl_prescription' AS table_name, path, COUNT(*) AS dup_count
  FROM hc_regulatory_sandbox.bronze_ingestion.spl_labels_prescription
  GROUP BY path HAVING COUNT(*) > 1
  UNION ALL
  SELECT 'spl_otc', path, COUNT(*)
  FROM hc_regulatory_sandbox.bronze_ingestion.spl_labels_otc
  GROUP BY path HAVING COUNT(*) > 1
  UNION ALL
  SELECT 'spl_animal', path, COUNT(*)
  FROM hc_regulatory_sandbox.bronze_ingestion.spl_labels_animal
  GROUP BY path HAVING COUNT(*) > 1
  UNION ALL
  SELECT 'spl_other', path, COUNT(*)
  FROM hc_regulatory_sandbox.bronze_ingestion.spl_labels_other
  GROUP BY path HAVING COUNT(*) > 1
)
SELECT 
  table_name,
  path,
  dup_count,
  '⚠️ ACTION REQUIRED: Deduplicate table' AS recommendation
FROM duplicates
ORDER BY dup_count DESC, table_name;

In [0]:
%sql
-- ============================================================================
-- 8. EXECUTIVE SUMMARY & RECOMMENDATIONS
-- ============================================================================
SELECT 
  '📋 BRONZE LAYER DATA QUALITY REPORT' AS section,
  '=================================================================================' AS divider,
  '' AS blank1,
  '📊 INGESTION SUMMARY:' AS metrics_header,
  '  • Total Tables: 6 active (7 including obsolete)' AS metric1,
  '  • Total Rows: ~3.8M rows ingested' AS metric2,
  '  • CSV Tables: 2 (fda_master: 946K, providers: 2.86M)' AS metric3,
  '  • XML Tables: 4 (110 files stored as binary)' AS metric4,
  '' AS blank2,
  '⚠️  CRITICAL ISSUES FOUND:' AS issues_header,
  '  1. DUPLICATE ROWS in XML tables (otc, animal, other)' AS issue1,
  '     - Likely caused by re-running ingestion without clearing checkpoints' AS issue1_detail,
  '     - ACTION: Deduplicate using QUALIFY ROW_NUMBER() OVER (PARTITION BY path)' AS issue1_action,
  '' AS blank3,
  '  2. OBSOLETE TABLE: clinical_labels (old naming convention)' AS issue2,
  '     - ACTION: DROP TABLE hc_regulatory_sandbox.bronze_ingestion.clinical_labels' AS issue2_action,
  '' AS blank4,
  '✓ DATA QUALITY STATUS:' AS quality_header,
  '  • Schema validation: PASS (column names cleaned correctly)' AS check1,
  '  • Binary content: PASS (XML files stored with path, length, content)' AS check2,
  '  • NULL checks: PASS (no NULL paths or empty content)' AS check3,
  '  • File integrity: PASS (all files have content)' AS check4,
  '' AS blank5,
  '🛠️  RECOMMENDED ACTIONS (Priority Order):' AS actions_header,
  '  1. [HIGH] Deduplicate XML tables (spl_otc, spl_animal, spl_other)' AS action1,
  '  2. [MEDIUM] Drop obsolete clinical_labels table' AS action2,
  '  3. [LOW] Document expected row counts in metadata' AS action3,
  '  4. [NEXT] Create silver layer notebook for XML parsing' AS action4
LIMIT 1;

In [0]:
%sql
-- ============================================================================
-- 9. UNIFIED CLEANUP SCRIPT
-- ============================================================================
-- PURPOSE: Sanitize Bronze layer by removing ghost records and obsolete tables
-- SAFE FOR: Free tier (uses conservative VACUUM retention)
--
-- ISSUE: NULL placeholder records from streaming micro-batches that finished 
--        with no data (common when trigger runs after all files processed)
-- ============================================================================

-- STEP 1: Purge NULL placeholder records from XML tables
DELETE FROM hc_regulatory_sandbox.bronze_ingestion.spl_labels_otc 
WHERE path IS NULL;

DELETE FROM hc_regulatory_sandbox.bronze_ingestion.spl_labels_animal 
WHERE path IS NULL;

DELETE FROM hc_regulatory_sandbox.bronze_ingestion.spl_labels_other 
WHERE path IS NULL;

-- STEP 2: Drop the legacy/obsolete table (old naming convention)
DROP TABLE IF EXISTS hc_regulatory_sandbox.bronze_ingestion.clinical_labels;

-- STEP 3: Optimize cleaned tables to reclaim space
-- NOTE: Using RETAIN 168 HOURS (7 days) for free-tier safety
--       (Change to 0 HOURS in production if aggressive cleanup needed)
VACUUM hc_regulatory_sandbox.bronze_ingestion.spl_labels_otc RETAIN 168 HOURS;
VACUUM hc_regulatory_sandbox.bronze_ingestion.spl_labels_animal RETAIN 168 HOURS;
VACUUM hc_regulatory_sandbox.bronze_ingestion.spl_labels_other RETAIN 168 HOURS;

SELECT 
  '✅ CLEANUP COMPLETE' AS status,
  '  • NULL records removed from XML tables' AS step1,
  '  • Obsolete clinical_labels table dropped' AS step2,
  '  • Tables optimized with VACUUM (7-day retention)' AS step3,
  '  • Run Cell 1 to validate row counts' AS next_step;

In [0]:
%sql
-- ============================================================================
-- 7. TABLE METADATA & LINEAGE
-- ============================================================================
-- Check table creation/modification timestamps, identify obsolete tables
SELECT 
  table_name,
  table_type,
  data_source_format,
  created,
  last_altered,
  DATEDIFF(HOUR, created, last_altered) AS hours_since_creation,
  table_owner,
  CASE 
    WHEN table_name NOT IN ('fda_master', 'providers', 'spl_labels_prescription', 
                            'spl_labels_otc', 'spl_labels_animal', 'spl_labels_other') 
    THEN '⚠️ Obsolete - Consider dropping'
    ELSE '✓ Active'
  END AS table_status
FROM hc_regulatory_sandbox.information_schema.tables 
WHERE table_schema = 'bronze_ingestion'
ORDER BY last_altered DESC;

In [0]:
%sql
-- ============================================================================
-- 6. FILE SIZE STATISTICS: XML TABLES
-- ============================================================================
-- Analyze XML file size distribution (helps with downstream processing planning)
WITH file_stats AS (
  SELECT 'spl_prescription' AS table_name, path, length FROM hc_regulatory_sandbox.bronze_ingestion.spl_labels_prescription
  UNION ALL
  SELECT 'spl_otc', path, length FROM hc_regulatory_sandbox.bronze_ingestion.spl_labels_otc
  UNION ALL
  SELECT 'spl_animal', path, length FROM hc_regulatory_sandbox.bronze_ingestion.spl_labels_animal
  UNION ALL
  SELECT 'spl_other', path, length FROM hc_regulatory_sandbox.bronze_ingestion.spl_labels_other
)
SELECT 
  table_name,
  COUNT(*) AS file_count,
  ROUND(MIN(length) / 1024.0, 2) AS min_size_kb,
  ROUND(AVG(length) / 1024.0, 2) AS avg_size_kb,
  ROUND(MAX(length) / 1024.0, 2) AS max_size_kb,
  ROUND(SUM(length) / 1024.0 / 1024.0, 2) AS total_size_mb,
  ROUND(PERCENTILE(length, 0.5) / 1024.0, 2) AS median_size_kb
FROM file_stats
GROUP BY table_name
ORDER BY total_size_mb DESC;

In [0]:
%sql
-- ============================================================================
-- 5. NULL & DATA COMPLETENESS CHECK
-- ============================================================================
-- XML Tables: Check for NULL paths or empty content
WITH xml_nulls AS (
  SELECT 
    'spl_prescription' AS table_name,
    COUNT(*) AS total_rows,
    SUM(CASE WHEN path IS NULL THEN 1 ELSE 0 END) AS null_paths,
    SUM(CASE WHEN content IS NULL THEN 1 ELSE 0 END) AS null_content,
    SUM(CASE WHEN length = 0 OR length IS NULL THEN 1 ELSE 0 END) AS zero_length
  FROM hc_regulatory_sandbox.bronze_ingestion.spl_labels_prescription
  UNION ALL
  SELECT 'spl_otc', COUNT(*), 
    SUM(CASE WHEN path IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN content IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN length = 0 OR length IS NULL THEN 1 ELSE 0 END)
  FROM hc_regulatory_sandbox.bronze_ingestion.spl_labels_otc
  UNION ALL
  SELECT 'spl_animal', COUNT(*),
    SUM(CASE WHEN path IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN content IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN length = 0 OR length IS NULL THEN 1 ELSE 0 END)
  FROM hc_regulatory_sandbox.bronze_ingestion.spl_labels_animal
  UNION ALL
  SELECT 'spl_other', COUNT(*),
    SUM(CASE WHEN path IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN content IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN length = 0 OR length IS NULL THEN 1 ELSE 0 END)
  FROM hc_regulatory_sandbox.bronze_ingestion.spl_labels_other
)
SELECT 
  table_name,
  total_rows,
  null_paths,
  null_content,
  zero_length,
  CASE 
    WHEN null_paths + null_content + zero_length = 0 THEN '✓ Clean'
    ELSE '⚠️ Issues Found'
  END AS data_quality_status
FROM xml_nulls;